In [1]:
import os
import asyncio
from typing import Any
from mcp.server.fastmcp import FastMCP
from langchain.agents import create_agent 
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from dotenv import load_dotenv

load_dotenv()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
# Initialize LLM
model = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_API_BASE"),
    temperature=0
)

# Configure MCP servers
client = MultiServerMCPClient(
    {
        "calculator": {
            "command": "python",
            "args": ["/root/code/mcp_servers/calculator.py"],
            "transport": "stdio",
        },
        "weather": {
            "command": "python",
            "args": ["/root/code/mcp_servers/weather_server.py"],
            "transport": "stdio",
        }
    }
)

async def run_agent_with_mcp():
    """Run agent with real MCP servers using JSON-RPC calls"""
    
    # Load MCP tools
    tools = await client.get_tools()
    for t in tools:
        print(t.name)
    print(f"✅ Loaded {len(tools)} tools from MCP servers")

    # Create agent
    agent = create_agent(model, tools)

    print("\n=== TEST 1: Calculator add ===")
    calc_response = await agent.ainvoke({
        "tool_inputs": {"add": {"a": 42, "b": 58}}
    })
    print(f"Response: {calc_response['messages'][-1].content}")

    print("\n=== TEST 2: Calculator multiply ===")
    multiply_response = await agent.ainvoke({
        "tool_inputs": {"multiply": {"a": 8, "b": 9}}
    })
    print(f"Response: {multiply_response['messages'][-1].content}")

    print("\n=== TEST 3: Calculator divide ===")
    divide_response = await agent.ainvoke({
        "tool_inputs": {"divide": {"a": 100, "b": 4}}
    })
    print(f"Response: {divide_response['messages'][-1].content}")

    print("\n=== TEST 4: Weather MCP (example) ===")
    weather_response = await agent.ainvoke({
        "tool_inputs": {"get_weather": {"city": "New York"}}
    })
    print(f"Response: {weather_response['messages'][-1].content}")

# Run async function in Jupyter or script
await run_agent_with_mcp()